# Checkpoint Decision Matrix (Runnable)

This notebook is scenario-first: each row in the matrix has a runnable cell.

Goal:
- Quickly answer "what should I do in this case?"
- Show both outcome and recommended API call


## Matrix

| Scenario | Inputs | Expected outcome | What you do |
|---|---|---|---|
| No checkpointer | any `workflow_id` behavior | no lineage persistence/resume | use plain `SyncRunner()` |
| Checkpointer + no `workflow_id` | run once | auto-generated `workflow_id` | read `result.workflow_id` |
| Same ID + completed | resume call | `WorkflowAlreadyCompletedError` | branch from checkpoint |
| Same ID + failed | resume, no new values | allowed resume | rerun same ID |
| Same ID + paused | provide interrupt response | resume to completed | rerun same ID + response key |
| Same ID + new values | override values | `InputOverrideRequiresForkError` | branch from checkpoint |
| Same ID + structural graph change | changed topology/signatures | `GraphChangedError` | branch from checkpoint |
| Need clean branch | source workflow + optional superstep | new lineage | `cp.checkpoint(...)` + `runner.run(..., checkpoint=..., workflow_id=...)` |
| Nested graph changed | changed inner graph structure | `GraphChangedError` | branch outer workflow from checkpoint |


In [ ]:
from pathlib import Path

from hypergraph import END, AsyncRunner, Graph, SyncRunner, ifelse, interrupt, node
from hypergraph.checkpointers import SqliteCheckpointer
from hypergraph.exceptions import (
    GraphChangedError,
    InputOverrideRequiresForkError,
    WorkflowAlreadyCompletedError,
)

DB = Path("./tmp/checkpoint-decision-matrix.db")
# remove the file first from drive:
for p in [DB, Path(str(DB) + "-shm"), Path(str(DB) + "-wal")]:
    p.unlink(missing_ok=True)

DB.parent.mkdir(parents=True, exist_ok=True)
for p in [DB, Path(str(DB) + "-shm"), Path(str(DB) + "-wal")]:
    p.unlink(missing_ok=True)

cp = SqliteCheckpointer(str(DB))
runner = SyncRunner(checkpointer=cp)

In [ ]:
@node(output_name="doubled")
def double(x: int) -> int:
    return x * 2


@node(output_name="tripled")
def triple(doubled: int) -> int:
    return doubled * 3


base_graph = Graph([double, triple])

In [ ]:
base_graph

In [ ]:
print(DB.resolve())

## A/B: Without checkpointer vs with checkpointer


In [ ]:
plain_runner = SyncRunner()
plain_result = plain_runner.run(base_graph, {"x": 5}, workflow_id="plain-1")
print("without checkpointer:", plain_result.workflow_id)

In [ ]:
with_cp_result = runner.run(base_graph, {"x": 5})
print("with checkpointer auto-id:", with_cp_result.workflow_id)

In [ ]:
cp

In [ ]:
print("persisted run exists:", cp.get_run(with_cp_result.workflow_id) is not None)

## Completed workflow is terminal on same ID


In [ ]:
runner.run(base_graph, {"x": 3}, workflow_id="wf-complete")

In [ ]:
try:
    runner.run(base_graph, workflow_id="wf-complete")
except WorkflowAlreadyCompletedError as e:
    print(type(e).__name__)

## Failed workflow is resumable on same ID (no new values)


In [ ]:
switch = {"fail": True}


@node(output_name="seed")
def seed(x: int) -> int:
    return x


@node(output_name="out")
def flaky(seed: int) -> int:
    if switch["fail"]:
        raise RuntimeError("transient")
    return seed * 10

In [ ]:
flaky_graph = Graph([seed, flaky])

In [ ]:
flaky_graph

In [ ]:
first = runner.run(flaky_graph, {"x": 7}, workflow_id="wf-failed", error_handling="continue", override_workflow=True)

In [ ]:
print("first status:", first.status.value)

In [ ]:
first

In [ ]:
cp

In [ ]:
switch["fail"] = False

In [ ]:
forked = runner.run(flaky_graph, workflow_id="wf-failed", override_workflow=True)
print("forked workflow id:", forked.workflow_id)
print("forked out:", forked["out"])

In [ ]:
print("forked status:", forked.status.value, "| out =", forked["out"])

In [ ]:
forked

In [ ]:
cp.steps("wf-failed")

## Same ID + input override requires fork


In [ ]:
runner.run(base_graph, {"x": 4}, workflow_id="wf-override")

try:
    runner.run(base_graph, {"x": 100}, workflow_id="wf-override")
except InputOverrideRequiresForkError as e:
    print(type(e).__name__)

In [ ]:
fork_cp = cp.checkpoint("wf-override")
forked = runner.run(base_graph, {"x": 100}, checkpoint=fork_cp, workflow_id="wf-override-fork")
print("forked result:", forked.workflow_id, forked["tripled"])
cp.lineage("wf-override-fork")

In [ ]:
cp

## Same ID + structural graph change requires fork


In [ ]:
graph_v1 = Graph([double])
graph_v2 = Graph([double, triple])

runner.run(graph_v1, {"x": 2}, workflow_id="wf-graph")

try:
    runner.run(graph_v2, workflow_id="wf-graph")
except GraphChangedError as e:
    print(type(e).__name__)

graph_cp = cp.checkpoint("wf-graph")
forked = runner.run(graph_v2, {"x": 2}, checkpoint=graph_cp, workflow_id="wf-graph-v2")
print("forked graph change output:", forked["tripled"])

In [ ]:
forked.workflow_id

In [ ]:
cp

## First-class retry lineage


In [ ]:
# Explicit retry lineage.
retry_resume = runner.run(flaky_graph, retry_from="wf-failed")
retry_run = cp.get_run(retry_resume.workflow_id)
print("retry workflow id:", retry_resume.workflow_id)
print("retry_of:", retry_run.retry_of)
print("retry_index:", retry_run.retry_index)
print("retry out:", retry_resume["out"])

## Gate values are in state


In [ ]:
@node(output_name="next_count")
def increment(count: int) -> int:
    return count + 1


@ifelse(when_true=END, when_false="increment")
def done(next_count: int) -> bool:
    return next_count >= 2


gate_graph = Graph([increment, done])
runner.run(gate_graph, {"count": 2}, workflow_id="wf-gate")
state = cp.state("wf-gate")
print("_done in state:", "_done" in state, "value:", state.get("_done"))

## Nested graph changed: detect + fork


In [ ]:
@node(output_name="inner_out")
def inner_v1(x: int) -> int:
    return x + 1


@node(output_name="inner_mid")
def inner_v2_mid(x: int) -> int:
    return x + 2


@node(output_name="inner_out")
def inner_v2_end(inner_mid: int) -> int:
    return inner_mid * 2


@node(output_name="final")
def finish(inner_out: int) -> int:
    return inner_out + 10


outer_v1 = Graph([Graph([inner_v1], name="inner").as_node(name="nested"), finish])
outer_v2 = Graph([Graph([inner_v2_mid, inner_v2_end], name="inner").as_node(name="nested"), finish])

runner.run(outer_v1, {"x": 5}, workflow_id="wf-nested")
print("child exists:", cp.get_run("wf-nested/nested") is not None)

try:
    runner.run(outer_v2, workflow_id="wf-nested")
except GraphChangedError as e:
    print(type(e).__name__)

nested_cp = cp.checkpoint("wf-nested")
r = runner.run(outer_v2, {"x": 5}, checkpoint=nested_cp, workflow_id="wf-nested-v2")
print("forked nested output:", r["final"])
cp.lineage("wf-nested-v2")

## Paused workflow (interrupt) is resumable on same ID


In [ ]:
await cp.initialize()
async_runner = AsyncRunner(checkpointer=cp)


@interrupt(output_name="decision")
def approval() -> str: ...


@node(output_name="final_decision")
def finalize(decision: str) -> str:
    return f"decision={decision}"


paused_graph = Graph([approval, finalize])
paused = await async_runner.run(paused_graph, workflow_id="wf-paused")
print("paused status:", paused.status.value)

resumed = await async_runner.run(
    paused_graph,
    {paused.pause.response_key: "approve"},
    workflow_id="wf-paused",
)
print("resumed status:", resumed.status.value, "| value:", resumed["final_decision"])

## Inspect lineage quickly


In [ ]:
for run in cp.runs(limit=50):
    if run.id.startswith("wf-"):
        print(
            run.id,
            run.status.value,
            "forked_from=",
            run.forked_from,
            "fork_superstep=",
            run.fork_superstep,
        )

In [ ]:
# Optional cleanup
# await cp.close()
# for p in [DB, Path(str(DB) + '-shm'), Path(str(DB) + '-wal')]:
p.unlink(missing_ok=True)

In [ ]:
# Close checkpointer connection (recommended before reruns)
await cp.close()